In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("../data/vanilla_convertibles_data_enhanced.csv")

In [ ]:
df = df.dropna(subset=["price_normalized"])

X = df.drop(columns=["price_convertible", "price_normalized", "redemption"])
y = df[["price_normalized"]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42)


In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing._data import BOUNDS_THRESHOLD
from scipy.stats import norm

scaler_X = QuantileTransformer(output_distribution="normal", random_state=42)
scaler_y = QuantileTransformer(output_distribution="normal", random_state=42)

X_train = scaler_X.fit_transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train)
y_val = scaler_y.transform(y_val)

_lower = norm.ppf(BOUNDS_THRESHOLD)
_upper = norm.ppf(1 - BOUNDS_THRESHOLD)
X_train = np.clip(X_train, _lower, _upper)
X_val = np.clip(X_val, _lower, _upper)
X_test = np.clip(X_test, _lower, _upper)
y_train = np.clip(y_train, _lower, _upper)
y_val = np.clip(y_val, _lower, _upper)

In [ ]:
import matplotlib.pyplot as plt

n_estimators_range = [10, 25, 50, 100, 150, 200, 300, 400, 500, 750, 1000, 1250, 1500]
val_rmses = []

for n in n_estimators_range:
    rf_sweep = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rf_sweep.fit(X_train, y_train)
    y_pred = rf_sweep.predict(X_val)
    val_rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))
    print(f"n_estimators={n:>5d} | Val RMSE: {val_rmses[-1]:.6f}")

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, val_rmses, marker="o")
plt.xlabel("n_estimators")
plt.ylabel("Validation RMSE")
plt.title("Random Forest — n_estimators Elbow Analysis")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_

y_pred_val = best_rf.predict(X_val)
y_pred_test = best_rf.predict(X_test)

print("--- Validation ---")
print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred_val)))
print("MAE:", mean_absolute_error(y_val, y_pred_val))
print("R^2:", r2_score(y_val, y_pred_val))

print("\n--- Test ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
print("MAE:", mean_absolute_error(y_test, y_pred_test))
print("R^2:", r2_score(y_test, y_pred_test))